In [1]:
# ============================================
# 0. 라이브러리 import & DB 설정
# ============================================
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

# ============================================
# 1. DB 접속 정보
# ============================================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg=DB_CONFIG):
    pw = urllib.parse.quote_plus(cfg["password"])
    conn_str = (
        f"postgresql+psycopg2://{cfg['user']}:{pw}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(conn_str)

engine = get_engine()

# ============================================
# 2. 데이터 로드
# ============================================
SCHEMA = "b1_sparepart_usage"
TABLE  = "sparepart_usage"

query = f"""
SELECT *
FROM {SCHEMA}.{TABLE}
"""

df = pd.read_sql(query, engine)

print("[INFO] Raw shape:", df.shape)

# ============================================
# 3. 조건 1)
# end_day, dayornight 이외 컬럼이 모두 0인 행 제거
# ============================================
base_cols = ["end_day", "dayornight"]
value_cols = [c for c in df.columns if c not in base_cols]

df = df.loc[(df[value_cols] != 0).any(axis=1)].copy()

print("[INFO] After zero-row filter:", df.shape)

# ============================================
# 4. 조건 2)
# 컬럼 분리 → station / sparepart
# ============================================
melt_df = df.melt(
    id_vars=base_cols,
    value_vars=value_cols,
    var_name="raw_col",
    value_name="value"
)

# 0인 값 제거
melt_df = melt_df[melt_df["value"] != 0].copy()

# ============================================
# 5. station / sparepart 파싱
# ============================================
def parse_station_sparepart(col):
    """
    FCT1_mini_b  -> (FCT1, mini_b)
    FCT1_power   -> (FCT1, probe)
    """
    station, part = col.split("_", 1)
    if part == "power":
        part = "probe"
    return station, part

parsed = melt_df["raw_col"].apply(parse_station_sparepart)
melt_df["station"] = parsed.apply(lambda x: x[0])
melt_df["sparepart"] = parsed.apply(lambda x: x[1])

# ============================================
# 6. 최종 결과 DataFrame (end_day 오름차순 정렬)
# ============================================
result_df = (
    melt_df[["end_day", "dayornight", "station", "sparepart"]]
    .sort_values(by="end_day", ascending=True)
    .reset_index(drop=True)
)

result_df

# ============================================
# 7. 결과 확인
# ============================================
result_df

[INFO] Raw shape: (122, 18)
[INFO] After zero-row filter: (47, 18)


,end_day,dayornight,station,sparepart
0,20250804,night,FCT2,usb_a
1,20250804,night,FCT1,usb_c
2,20250804,night,FCT2,mini_b
3,20250804,night,FCT4,usb_a
4,20250804,night,FCT3,mini_b
...,...,...,...,...
243,20251031,night,FCT2,usb_c
244,20251031,night,FCT2,usb_a
245,20251031,night,FCT1,usb_c
246,20251031,night,FCT4,probe


In [2]:
# ============================================
# 4) end_day + dayornight → from_day/from_time/to_day/to_time 생성
# ============================================
import pandas as pd

# result_df 컬럼 확인: end_day, dayornight, station, sparepart
work_df = result_df.copy()

# end_day를 datetime으로 변환 (YYYYMMDD)
work_df["end_day_dt"] = pd.to_datetime(work_df["end_day"].astype(str), format="%Y%m%d")

# day/night 기준 시간 정의
DAY_FROM   = pd.to_timedelta("08:30:00")
DAY_TO     = pd.to_timedelta("20:29:59")
NIGHT_FROM = pd.to_timedelta("20:30:00")
NIGHT_TO   = pd.to_timedelta("08:29:59")  # 다음날

# from_dt / to_dt 생성
is_day = work_df["dayornight"].str.lower().eq("day")
is_night = work_df["dayornight"].str.lower().eq("night")

work_df["from_dt"] = pd.NaT
work_df["to_dt"]   = pd.NaT

# day: d-day 08:30:00 ~ 20:29:59
work_df.loc[is_day, "from_dt"] = work_df.loc[is_day, "end_day_dt"] + DAY_FROM
work_df.loc[is_day, "to_dt"]   = work_df.loc[is_day, "end_day_dt"] + DAY_TO

# night: d-day 20:30:00 ~ (d+1)-day 08:29:59
work_df.loc[is_night, "from_dt"] = work_df.loc[is_night, "end_day_dt"] + NIGHT_FROM
work_df.loc[is_night, "to_dt"]   = (work_df.loc[is_night, "end_day_dt"] + pd.Timedelta(days=1)) + NIGHT_TO

# from_day/from_time/to_day/to_time 분리 (표시 형식: YYYYMMDD, HH:MM:SS)
work_df["from_day"]  = work_df["from_dt"].dt.strftime("%Y%m%d")
work_df["from_time"] = work_df["from_dt"].dt.strftime("%H:%M:%S")
work_df["to_day"]    = work_df["to_dt"].dt.strftime("%Y%m%d")
work_df["to_time"]   = work_df["to_dt"].dt.strftime("%H:%M:%S")

# ============================================
# 5) 첨부파일 형태로 출력
# (station, sparepart, from_day, from_time, to_day, to_time)
# ============================================
out_df = (
    work_df[["station", "sparepart", "from_day", "from_time", "to_day", "to_time"]]
    .sort_values(by=["station", "sparepart", "from_day", "from_time"], ascending=True)
    .reset_index(drop=True)
)

out_df

,station,sparepart,from_day,from_time,to_day,to_time
0,FCT1,mini_b,20250804,20:30:00,20250805,08:29:59
1,FCT1,mini_b,20250806,20:30:00,20250807,08:29:59
2,FCT1,mini_b,20250808,20:30:00,20250809,08:29:59
3,FCT1,mini_b,20250812,08:30:00,20250812,20:29:59
4,FCT1,mini_b,20250814,08:30:00,20250814,20:29:59
...,...,...,...,...,...,...
243,FCT4,usb_c,20250915,08:30:00,20250915,20:29:59
244,FCT4,usb_c,20250917,08:30:00,20250917,20:29:59
245,FCT4,usb_c,20250925,20:30:00,20250926,08:29:59
246,FCT4,usb_c,20250929,20:30:00,20250930,08:29:59


In [3]:
# ============================================
# 6) (from~to) 시간범위 내 Manual→Auto(>=3분) 구간 찾기
#    - NA 행 제외 버전
# ============================================
import pandas as pd
from sqlalchemy import text

SCHEMA_ML = "d1_machine_log"

TABLE_BY_STATION = {
    "FCT1": "FCT1_machine_log",
    "FCT2": "FCT2_machine_log",
    "FCT3": "FCT3_machine_log",
    "FCT4": "FCT4_machine_log",
}

# ---- 1) 시간 윈도우 datetime 변환 ----
df_win = out_df.copy()

df_win["from_dt"] = pd.to_datetime(
    df_win["from_day"] + " " + df_win["from_time"],
    format="%Y%m%d %H:%M:%S"
)
df_win["to_dt"] = pd.to_datetime(
    df_win["to_day"] + " " + df_win["to_time"],
    format="%Y%m%d %H:%M:%S"
)

df_win["station"] = df_win["station"].astype(str).str.strip()

# 결과 컬럼
for c in ["manual_day", "manual_time", "auto_day", "auto_time"]:
    df_win[c] = pd.NA

# ---- 2) station별 머신로그 조회 ----
for st, g in df_win.groupby("station", sort=False):
    if st not in TABLE_BY_STATION:
        continue

    table = TABLE_BY_STATION[st]

    min_dt = g["from_dt"].min()
    max_dt = g["to_dt"].max()

    min_day = int(min_dt.strftime("%Y%m%d"))
    max_day = int((max_dt + pd.Timedelta(days=1)).strftime("%Y%m%d"))

    sql = text(f"""
        SELECT
            end_day::text  AS end_day,
            end_time::text AS end_time,
            contents::text AS contents
        FROM {SCHEMA_ML}."{table}"
        WHERE
            end_day::int BETWEEN :min_day AND :max_day
            AND (
                contents ILIKE '%Manual mode%'
                OR contents ILIKE '%Auto mode%'
            )
        ORDER BY end_day, end_time
    """)

    ml = pd.read_sql(sql, engine, params={
        "min_day": min_day,
        "max_day": max_day
    })

    if ml.empty:
        continue

    ml["end_dt"] = pd.to_datetime(
        ml["end_day"] + " " + ml["end_time"],
        errors="coerce"
    )

    ml = ml[(ml["end_dt"] >= min_dt) & (ml["end_dt"] <= max_dt)]
    if ml.empty:
        continue

    ml["is_manual"] = ml["contents"].str.contains("Manual mode", case=False, na=False)
    ml["is_auto"]   = ml["contents"].str.contains("Auto mode",   case=False, na=False)

    manual_times = ml.loc[ml["is_manual"], "end_dt"].sort_values().reset_index(drop=True)
    auto_times   = ml.loc[ml["is_auto"],   "end_dt"].sort_values().reset_index(drop=True)

    if manual_times.empty or auto_times.empty:
        continue

    # ---- 3) 윈도우별 Manual → Auto(>=180초) 매칭 ----
    for idx, row in g.iterrows():
        w_from = row["from_dt"]
        w_to   = row["to_dt"]

        m_in = manual_times[(manual_times >= w_from) & (manual_times <= w_to)]
        if m_in.empty:
            continue

        for m_t in m_in:
            a_in = auto_times[(auto_times >= m_t) & (auto_times <= w_to)]
            if a_in.empty:
                continue

            a_t = a_in.iloc[0]
            if (a_t - m_t).total_seconds() >= 180:
                df_win.loc[idx, "manual_day"]  = m_t.strftime("%Y%m%d")
                df_win.loc[idx, "manual_time"] = m_t.strftime("%H:%M:%S")
                df_win.loc[idx, "auto_day"]    = a_t.strftime("%Y%m%d")
                df_win.loc[idx, "auto_time"]   = a_t.strftime("%H:%M:%S")
                break

# ---- 4) NA 행 완전 제거 + 최종 출력 ----
final_df = (
    df_win[[
        "station", "sparepart",
        "from_day", "from_time",
        "to_day", "to_time",
        "manual_day", "manual_time",
        "auto_day", "auto_time"
    ]]
    .dropna()   # ★ 핵심: NA 있는 행 전부 제거
    .sort_values(by=["station", "sparepart", "from_day", "from_time"])
    .reset_index(drop=True)
)

final_df

,station,sparepart,from_day,from_time,to_day,to_time,manual_day,manual_time,auto_day,auto_time
0,FCT1,mini_b,20250812,08:30:00,20250812,20:29:59,20250812,15:00:02,20250812,15:04:34
1,FCT1,mini_b,20250819,08:30:00,20250819,20:29:59,20250819,16:56:02,20250819,17:05:28
2,FCT1,mini_b,20250825,20:30:00,20250826,08:29:59,20250825,21:19:30,20250826,01:53:08
3,FCT1,mini_b,20250829,20:30:00,20250830,08:29:59,20250829,23:06:04,20250830,00:55:49
4,FCT1,mini_b,20250904,20:30:00,20250905,08:29:59,20250904,20:51:02,20250904,21:05:51
...,...,...,...,...,...,...,...,...,...,...
94,FCT4,usb_c,20250827,20:30:00,20250828,08:29:59,20250828,02:25:44,20250828,02:29:45
95,FCT4,usb_c,20250829,08:30:00,20250829,20:29:59,20250829,08:36:47,20250829,08:44:15
96,FCT4,usb_c,20250905,08:30:00,20250905,20:29:59,20250905,14:01:29,20250905,14:07:03
97,FCT4,usb_c,20250909,08:30:00,20250909,20:29:59,20250909,13:24:32,20250909,13:30:47


In [4]:
# ============================================
# 7) auto(현재행) ~ manual(다음행) 구간 생산량(amount) 계산
#    - DB: 100.105.75.47
#    - schema/table: a1_fct_vision_testlog_txt_processing_history.fct_vision_testlog_txt_processing_history
# ============================================
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse

# ---------- DB 접속 (요청 DB_CONFIG) ----------
DB_CONFIG_2 = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine2(cfg=DB_CONFIG_2):
    pw = urllib.parse.quote_plus(cfg["password"])
    conn_str = f"postgresql+psycopg2://{cfg['user']}:{pw}@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    return create_engine(conn_str, pool_pre_ping=True)

engine2 = get_engine2()

# ---------- 대상 테이블 (정정 반영) ----------
H_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
H_TABLE  = "fct_vision_testlog_txt_processing_history"

# ---------- 0) 입력 DF 준비 ----------
df_in = final_df.copy()

# station 필터 (FCT1~4만)
df_in["station"] = df_in["station"].astype(str).str.strip()
df_in = df_in[df_in["station"].isin(["FCT1", "FCT2", "FCT3", "FCT4"])].copy()

# datetime 생성
df_in["manual_dt"] = pd.to_datetime(
    df_in["manual_day"] + " " + df_in["manual_time"],
    format="%Y%m%d %H:%M:%S",
    errors="coerce",
)
df_in["auto_dt"] = pd.to_datetime(
    df_in["auto_day"] + " " + df_in["auto_time"],
    format="%Y%m%d %H:%M:%S",
    errors="coerce",
)

# 정렬 (station+sparepart 내 시간순)
df_in = df_in.sort_values(by=["station", "sparepart", "manual_dt", "auto_dt"]).reset_index(drop=True)

# ---------- 1) 현재 auto_dt, 다음 manual_dt 로 구간 생성 ----------
df_in["next_manual_dt"] = df_in.groupby(["station", "sparepart"])["manual_dt"].shift(-1)

ranges_df = df_in.loc[
    df_in["auto_dt"].notna() & df_in["next_manual_dt"].notna()
].copy()

ranges_df["from_dt"] = ranges_df["auto_dt"]
ranges_df["to_dt"]   = ranges_df["next_manual_dt"]

# from < to 인 것만
ranges_df = ranges_df[ranges_df["to_dt"] > ranges_df["from_dt"]].copy()

# 출력용 분해
ranges_df["from_day"]  = ranges_df["from_dt"].dt.strftime("%Y%m%d")
ranges_df["from_time"] = ranges_df["from_dt"].dt.strftime("%H:%M:%S")
ranges_df["to_day"]    = ranges_df["to_dt"].dt.strftime("%Y%m%d")
ranges_df["to_time"]   = ranges_df["to_dt"].dt.strftime("%H:%M:%S")

# ---------- 2) 히스토리 테이블 컬럼 읽기 (정정 스키마 반영) ----------
with engine2.begin() as conn:
    cols = pd.read_sql(
        text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = :s AND table_name = :t
            ORDER BY ordinal_position
        """),
        conn,
        params={"s": H_SCHEMA, "t": H_TABLE}
    )["column_name"].tolist()

cols_set = set(cols)

# station 컬럼 필수
if "station" not in cols_set:
    raise ValueError(f"[ERROR] history 테이블에 station 컬럼이 없습니다. 테이블 컬럼: {cols}")

# 시간 컬럼 후보 자동 감지
# 1순위: end_day + end_time
# 2순위: day + end_time
# 3순위: end_day + time
if {"end_day", "end_time"}.issubset(cols_set):
    day_col, time_col = "end_day", "end_time"
elif {"day", "end_time"}.issubset(cols_set):
    day_col, time_col = "day", "end_time"
elif {"end_day", "time"}.issubset(cols_set):
    day_col, time_col = "end_day", "time"
else:
    raise ValueError(f"[ERROR] 시간 컬럼 조합을 찾지 못했습니다. 테이블 컬럼: {cols}")

# ---------- 3) amount 계산 (시간범위 내 행 count) ----------
sql_count = text(f"""
    SELECT COUNT(*)::bigint AS cnt
    FROM {H_SCHEMA}."{H_TABLE}"
    WHERE station = :station
      AND ({day_col}::text || ' ' || {time_col}::text)::timestamp
            BETWEEN :from_dt AND :to_dt
      AND {day_col}::int BETWEEN :from_day AND :to_day
""")

amounts = []
with engine2.begin() as conn:
    for r in ranges_df.itertuples(index=False):
        from_day_i = int(r.from_day)
        to_day_i   = int(r.to_day)

        cnt = conn.execute(
            sql_count,
            {
                "station": r.station,
                "from_dt": r.from_dt.to_pydatetime(),
                "to_dt":   r.to_dt.to_pydatetime(),
                "from_day": from_day_i,
                "to_day":   to_day_i,
            }
        ).scalar_one()

        amounts.append(int(cnt))

ranges_df["amount"] = amounts

# ---------- 4) amount 필터 (0 제외 + 1000 이상만) + 첨부 형식 출력 ----------
out7_df = (
    ranges_df[["station", "sparepart", "from_day", "from_time", "to_day", "to_time", "amount"]]
    .query("amount >= 1000")
    .sort_values(by=["station", "sparepart", "from_day", "from_time"])
    .reset_index(drop=True)
)

out7_df

,station,sparepart,from_day,from_time,to_day,to_time,amount
0,FCT1,mini_b,20250812,15:04:34,20250819,16:56:02,10291
1,FCT1,mini_b,20250819,17:05:28,20250825,21:19:30,6465
2,FCT1,mini_b,20250826,01:53:08,20250829,23:06:04,6927
3,FCT1,mini_b,20250830,00:55:49,20250904,20:51:02,7581
4,FCT1,mini_b,20250904,21:05:51,20250912,15:08:18,13211
...,...,...,...,...,...,...,...
79,FCT4,usb_c,20250821,10:46:55,20250828,02:25:44,8382
80,FCT4,usb_c,20250828,02:29:45,20250829,08:36:47,1997
81,FCT4,usb_c,20250829,08:44:15,20250905,14:01:29,10477
82,FCT4,usb_c,20250905,14:07:03,20250909,13:24:32,5730


In [7]:
# ============================================
# 8) 산출 결과 바탕화면 CSV 자동 저장
# ============================================
import os
from datetime import datetime

# 바탕화면 경로
DESKTOP = os.path.join(os.path.expanduser("~"), "Desktop")

# 파일명: amount_range_YYYYMMDD_HHMMSS.csv
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(DESKTOP, f"amount_range_{ts}.csv")

# 저장
out7_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"[OK] 결과 CSV 저장 완료 → {out_path}")

[OK] 결과 CSV 저장 완료 → C:\Users\user\Desktop\amount_range_20251231_175533.csv


In [5]:
# ============================================
# 8) sparepart별 amount 퍼센타일 분석 + plotly json 생성
#    - min, p10, p25, p50, p75, max (내림 정수)
#    - plotly_graph 컬럼에 json 저장
# ============================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# out7_df: station, sparepart, from_day, from_time, to_day, to_time, amount
df_amt = out7_df.copy()

# 방어: amount 숫자형 변환 + NA 제거
df_amt["amount"] = pd.to_numeric(df_amt["amount"], errors="coerce")
df_amt = df_amt.dropna(subset=["sparepart", "amount"]).copy()

def _floor_int(x):
    # 내림 정수 (NaN 방어)
    if pd.isna(x):
        return np.nan
    return int(np.floor(x))

rows = []
for sp, g in df_amt.groupby("sparepart", sort=True):
    vals = g["amount"].astype(float).values

    if len(vals) == 0:
        continue

    q_min = np.min(vals)
    q10   = np.percentile(vals, 10)
    q25   = np.percentile(vals, 25)
    q50   = np.percentile(vals, 50)
    q75   = np.percentile(vals, 75)
    q_max = np.max(vals)

    # ---- plotly 그래프 (distribution 확인용) ----
    fig = go.Figure()
    fig.add_trace(go.Box(y=vals, name=str(sp), boxpoints="outliers"))
    fig.update_layout(
        title=f"amount distribution - {sp}",
        xaxis_title="sparepart",
        yaxis_title="amount",
        showlegend=False,
        height=260,
        margin=dict(l=30, r=30, t=40, b=30),
    )

    rows.append({
        "sparepart": sp,
        "min": _floor_int(q_min),
        "p10": _floor_int(q10),
        "p25": _floor_int(q25),
        "p50": _floor_int(q50),
        "p75": _floor_int(q75),
        "max": _floor_int(q_max),
        "plotly_graph": fig.to_json()
    })

summary8_df = pd.DataFrame(rows)

# 첨부 형식처럼 정렬
summary8_df = summary8_df.sort_values(by="sparepart").reset_index(drop=True)

summary8_df

,sparepart,min,p10,p25,p50,p75,max,plotly_graph
0,mini_b,1271,3933,4477,6846,9774,18946,"{""data"":[{""boxpoints"":""outliers"",""name"":""mini_..."
1,usb_a,3603,4127,4922,7119,9774,18946,"{""data"":[{""boxpoints"":""outliers"",""name"":""usb_a..."
2,usb_c,1271,1945,4671,8410,15103,33010,"{""data"":[{""boxpoints"":""outliers"",""name"":""usb_c..."


In [6]:
# ============================================
# 9) summary8_df 저장
#    - DB: localhost
#    - schema: e3_sparepart_replacement
#    - table : sparepart_life_amount
# ============================================
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse

# ---------- DB 설정 ----------
DB_CONFIG_SAVE = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine_save(cfg=DB_CONFIG_SAVE):
    pw = urllib.parse.quote_plus(cfg["password"])
    conn_str = (
        f"postgresql+psycopg2://{cfg['user']}:{pw}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(conn_str, pool_pre_ping=True)

engine_save = get_engine_save()

TARGET_SCHEMA = "e3_sparepart_replacement"
TARGET_TABLE  = "sparepart_life_amount"

# ---------- 1) 스키마 생성 ----------
with engine_save.begin() as conn:
    conn.execute(text(f"""
        CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}
    """))

# ---------- 2) 테이블 생성 ----------
with engine_save.begin() as conn:
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {TARGET_SCHEMA}.{TARGET_TABLE} (
            sparepart      TEXT PRIMARY KEY,
            min            INTEGER,
            p10            INTEGER,
            p25            INTEGER,
            p50            INTEGER,
            p75            INTEGER,
            max            INTEGER,
            plotly_graph   TEXT
        )
    """))

# ---------- 3) UPSERT 저장 ----------
upsert_sql = text(f"""
    INSERT INTO {TARGET_SCHEMA}.{TARGET_TABLE}
    (sparepart, min, p10, p25, p50, p75, max, plotly_graph)
    VALUES
    (:sparepart, :min, :p10, :p25, :p50, :p75, :max, :plotly_graph)
    ON CONFLICT (sparepart)
    DO UPDATE SET
        min          = EXCLUDED.min,
        p10          = EXCLUDED.p10,
        p25          = EXCLUDED.p25,
        p50          = EXCLUDED.p50,
        p75          = EXCLUDED.p75,
        max          = EXCLUDED.max,
        plotly_graph = EXCLUDED.plotly_graph
""")

with engine_save.begin() as conn:
    for r in summary8_df.itertuples(index=False):
        conn.execute(
            upsert_sql,
            {
                "sparepart": r.sparepart,
                "min": r.min,
                "p10": r.p10,
                "p25": r.p25,
                "p50": r.p50,
                "p75": r.p75,
                "max": r.max,
                "plotly_graph": r.plotly_graph,
            }
        )

print("[INFO] sparepart_life_amount 저장 완료")

[INFO] sparepart_life_amount 저장 완료
